# Importação de Bibliotecas

In [ ]:
import sqlalchemy
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin


# Amostragem

In [2]:
database = sqlalchemy.create_engine("sqlite:///../../data/feature_store.db")


with open("abt.sql", "r") as f:
    query = f.read()

df = pd.read_sql(query, database)

In [3]:
df.head()

,DtRef,IdStore,QtdSales7d,QtdSales14d,QtdSales28d,QtdSales42d,QtdSales56d,QtdSales84d,AvgSales7d,AvgSales14d,...,DaysSchoolHoliday56d,DaysSchoolHoliday84d,DaysSchoolHolidayRate7d,DaysSchoolHolidayRate14d,DaysSchoolHolidayRate28d,DaysSchoolHolidayRate42d,DaysSchoolHolidayRate56d,DaysSchoolHolidayRate84d,MonthsSinceCompetition,Target
0,2013-03-26,1,35178.0,65480.0,130434.0,190894.0,255712.0,371108,5863.000000,5456.666667,...,1,12,0.142857,0.071429,0.035714,0.023810,0.017857,0.142857,54.0,168765.0
1,2013-03-26,2,34387.0,59946.0,118145.0,171940.0,227987.0,330911,5731.166667,4995.500000,...,6,10,0.142857,0.071429,0.035714,0.142857,0.107143,0.119048,64.0,161514.0
2,2013-03-26,3,49165.0,87382.0,170497.0,247326.0,330148.0,479824,8194.166667,7281.833333,...,1,5,0.142857,0.071429,0.035714,0.023810,0.017857,0.059524,75.0,237269.0
3,2013-03-26,4,60529.0,111777.0,227692.0,338044.0,458532.0,667726,10088.166667,9314.750000,...,6,10,0.142857,0.071429,0.035714,0.023810,0.107143,0.119048,42.0,321894.0
4,2013-03-26,5,30620.0,53256.0,105674.0,154992.0,206796.0,304004,5103.333333,4438.000000,...,10,12,0.000000,0.000000,0.000000,0.095238,0.178571,0.142857,-25.0,147260.0


In [4]:
df_sorted = df.sort_values('DtRef').reset_index(drop=True)

n = len(df_sorted)
n_train = int(0.6 * n)
n_val   = int(0.2 * n)
n_test  = n - n_train - n_val

train_df = df_sorted.iloc[:n_train]
val_df   = df_sorted.iloc[n_train:n_train + n_val]
test_df  = df_sorted.iloc[n_train + n_val:]

drop_cols = ['Target', 'IdStore', 'DtRef']

X_train = train_df.drop(columns=drop_cols, errors='ignore')
y_train = train_df['Target']

X_val = val_df.drop(columns=drop_cols, errors='ignore')
y_val = val_df['Target']

X_test = test_df.drop(columns=drop_cols, errors='ignore')
y_test = test_df['Target']

# Exploração

## Estatisticas

In [5]:
X_train_numeric = X_train.select_dtypes(include='number')

correlations = X_train_numeric.corrwith(y_train, method='pearson')

top10_positive = correlations.nlargest(10)

top10_negative = correlations.nsmallest(10)

c:\Users\patry\AppData\Local\miniconda3\envs\ds_env\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\patry\AppData\Local\miniconda3\envs\ds_env\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [6]:
top10_positive.head(10)


QtdSales84d           0.952284
QtdSales56d           0.949342
QtdSales42d           0.946698
QtdSales28d           0.942001
AvgSales84d           0.940410
AvgSales56d           0.935854
AvgSales42d           0.932304
AvgSalesNoPromo84d    0.930760
QtdSales14d           0.929409
AvgSalesNoPromo56d    0.928421
dtype: float64

In [7]:
top10_negative.head(10)

LiftSalesPromo56d   -0.198943
LiftSalesPromo84d   -0.197200
LiftSalesPromo42d   -0.192446
DaysClosed84d       -0.184697
LiftSalesPromo28d   -0.182942
DaysClosed56d       -0.170750
LiftSalesPromo14d   -0.166455
DaysClosed42d       -0.159019
DaysClosed28d       -0.150568
LiftSalesPromo7d    -0.148684
dtype: float64

In [8]:
with pd.option_context('display.max_columns', None, 'display.width', 1000, 'display.max_rows', None):
    display(X_train_numeric.describe().T)

,count,mean,std,min,25%,50%,75%,max
QtdSales7d,533047.0,39838.581855,16111.182947,0.000000,29175.000000,36998.000000,46789.000000,1.931100e+05
QtdSales14d,534307.0,79621.540947,30168.442738,0.000000,59891.000000,74630.000000,92548.000000,3.599820e+05
QtdSales28d,534996.0,159155.401816,59186.755058,0.000000,120263.000000,149657.000000,184466.000000,6.892920e+05
QtdSales42d,534996.0,238783.969065,88071.613250,0.000000,180879.000000,224741.000000,276472.000000,9.772930e+05
QtdSales56d,534996.0,318307.698693,116907.403940,0.000000,241470.000000,299907.500000,368256.000000,1.284628e+06
QtdSales84d,534996.0,476953.962385,174567.280092,0.000000,362422.000000,449824.000000,551549.000000,1.864543e+06
AvgSales7d,532354.0,6915.907215,2692.256248,0.000000,5097.833333,6433.333333,8110.482143,3.236800e+04
AvgSales14d,533857.0,6918.490009,2539.000316,0.000000,5218.545455,6485.300000,8020.833333,2.845500e+04
AvgSales28d,534710.0,6901.123667,2487.417376,0.000000,5231.052083,6496.133399,7992.000000,2.609965e+04
AvgSales42d,534743.0,6890.353137,2464.923319,1426.000000,5236.264706,6494.027778,7975.986111,2.603736e+04


In [9]:
# Tabela de frequência relativa das variáveis categóricas StoreType e Assortment
print("Frequência relativa de StoreType:")
print(X_train['StoreType'].value_counts(normalize=True, dropna=False))
print("\nFrequência relativa de Assortment:")
print(X_train['Assortment'].value_counts(normalize=True, dropna=False))

Frequência relativa de StoreType:
StoreType
a    0.539903
d    0.312109
c    0.132743
b    0.015245
Name: proportion, dtype: float64

Frequência relativa de Assortment:
Assortment
a    0.531873
c    0.460056
b    0.008071
Name: proportion, dtype: float64


## Tipos

In [10]:
# Mostrar as colunas cujas variáveis são do tipo object
object_cols = X_train.select_dtypes(include='object').columns.tolist()
print("Colunas do tipo object:", object_cols)

Colunas do tipo object: ['StoreType', 'Assortment']


In [13]:
X_train['SalesGrowth42dYearAgo'] = pd.to_numeric(X_train['SalesGrowth42dYearAgo'], errors='coerce')
print("Tipo após conversão:", X_train['SalesGrowth42dYearAgo'].dtype)
print("Valores NaN após conversão:", X_train['SalesGrowth42dYearAgo'].isna().sum())

Tipo após conversão: float64
Valores NaN após conversão: 316046


## NaN

In [ ]:
nan_pct = X_train.isna().mean() * 100

nan_pct = nan_pct[nan_pct > 0].sort_values(ascending=False)

nan_pct_df = nan_pct.to_frame('Porcentagem_NaN').reset_index().rename(columns={'index': 'Coluna'})

with pd.option_context('display.max_columns', None, 'display.width', 1000, 'display.max_rows', None):
    display(nan_pct_df)

,Coluna,Porcentagem_NaN
0,SalesGrowth42dYearAgo,59.074460
1,CompetitionOpen,31.748649
2,MonthsSinceCompetition,31.748649
3,LiftSalesPromo7d,22.429513
4,MinCustomersPromo7d,22.320728
5,AvgCustomersPromo7d,22.320728
6,MinSalesPromo7d,22.320728
7,AvgSalesPromo7d,22.320728
8,QtdSalesPromo7d,21.729882
9,MaxSalesPromo7d,21.729882


## Tratamento NaN

In [ ]:

class NullImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        # define columns and fill values for different groups
        self.lift_vars = [
            'LiftSalesPromo7d', 'LiftSalesPromo14d', 'LiftSalesPromo28d', 
            'LiftSalesPromo42d', 'LiftSalesPromo56d', 'LiftSalesPromo84d', 
            'MonthsSinceCompetition'
        ]
        self.cols_promo = [
            'SalesGrowth42dYearAgo_missing', 'QtdSalesPromo7d', 'QtdSalesPromo14d', 
            'QtdSalesPromo28d', 'QtdSalesPromo42d', 'QtdSalesPromo56d', 'QtdSalesPromo84d', 
            'AvgSalesPromo7d', 'AvgSalesPromo14d', 'AvgSalesPromo28d', 'AvgSalesPromo42d', 
            'AvgSalesPromo56d', 'AvgSalesPromo84d', 'MinSalesPromo7d', 'MinSalesPromo14d', 
            'MinSalesPromo28d', 'MinSalesPromo42d', 'MinSalesPromo56d', 'MinSalesPromo84d', 
            'MaxSalesPromo7d', 'MaxSalesPromo14d', 'MaxSalesPromo28d', 'MaxSalesPromo42d', 
            'MaxSalesPromo56d', 'MaxSalesPromo84d', 'QtdSalesNoPromo7d', 'QtdSalesNoPromo14d', 
            'QtdSalesNoPromo28d', 'QtdSalesNoPromo42d', 'QtdSalesNoPromo56d', 'QtdSalesNoPromo84d', 
            'AvgSalesNoPromo7d', 'AvgSalesNoPromo14d', 'AvgSalesNoPromo28d', 'AvgSalesNoPromo42d', 
            'AvgSalesNoPromo56d', 'AvgSalesNoPromo84d', 'MinSalesNoPromo7d', 'MinSalesNoPromo14d', 
            'MinSalesNoPromo28d', 'MinSalesNoPromo42d', 'MinSalesNoPromo56d', 'MinSalesNoPromo84d', 
            'MaxSalesNoPromo7d', 'MaxSalesNoPromo14d', 'MaxSalesNoPromo28d', 'MaxSalesNoPromo42d', 
            'MaxSalesNoPromo56d', 'MaxSalesNoPromo84d', 'QtdCustomersPromo7d', 'QtdCustomersPromo14d', 
            'QtdCustomersPromo28d', 'QtdCustomersPromo42d', 'QtdCustomersPromo56d', 'QtdCustomersPromo84d', 
            'AvgCustomersPromo7d', 'AvgCustomersPromo14d', 'AvgCustomersPromo28d', 'AvgCustomersPromo42d', 
            'AvgCustomersPromo56d', 'AvgCustomersPromo84d', 'MinCustomersPromo7d', 'MinCustomersPromo14d', 
            'MinCustomersPromo28d', 'MinCustomersPromo42d', 'MinCustomersPromo56d', 'MinCustomersPromo84d', 
            'MaxCustomersPromo7d', 'MaxCustomersPromo14d', 'MaxCustomersPromo28d', 'MaxCustomersPromo42d', 
            'MaxCustomersPromo56d', 'MaxCustomersPromo84d', 'QtdCustomersNoPromo7d', 'QtdCustomersNoPromo14d', 
            'QtdCustomersNoPromo28d', 'QtdCustomersNoPromo42d', 'QtdCustomersNoPromo56d', 'QtdCustomersNoPromo84d', 
            'AvgCustomersNoPromo7d', 'AvgCustomersNoPromo14d', 'AvgCustomersNoPromo28d', 'AvgCustomersNoPromo42d', 
            'AvgCustomersNoPromo56d', 'AvgCustomersNoPromo84d', 'MinCustomersNoPromo7d', 'MinCustomersNoPromo14d', 
            'MinCustomersNoPromo28d', 'MinCustomersNoPromo42d', 'MinCustomersNoPromo56d', 'MinCustomersNoPromo84d', 
            'MaxCustomersNoPromo7d', 'MaxCustomersNoPromo14d', 'MaxCustomersNoPromo28d', 'MaxCustomersNoPromo42d', 
            'MaxCustomersNoPromo56d', 'MaxCustomersNoPromo84d', 'Promo2Open', 'DaysPromo7d', 
            'DaysPromo14d', 'DaysPromo28d', 'DaysPromo42d', 'DaysPromo56d', 'DaysPromo84d', 
            'DaysNoPromo7d', 'DaysNoPromo14d', 'DaysNoPromo28d', 'DaysNoPromo42d', 'DaysNoPromo56d', 
            'DaysNoPromo84d', 'DaysPromoRate7d', 'DaysPromoRate14d', 'DaysPromoRate28d', 
            'DaysPromoRate42d', 'DaysPromoRate56d', 'DaysPromoRate84d'
        ]
        self.cols_sales_customers = [
            'QtdSales7d', 'QtdSales14d', 'AvgSales7d', 'AvgSales14d', 'AvgSales28d', 'AvgSales42d', 
            'AvgSales56d', 'AvgSales84d', 'MinSales7d', 'MinSales14d', 'MinSales28d', 'MinSales42d', 
            'MinSales56d', 'MinSales84d', 'MaxSales7d', 'MaxSales14d', 'MaxSales28d', 'MaxSales42d', 
            'MaxSales56d', 'MaxSales84d', 'Growth_AvgSales_7d_vs_28d', 'Growth_AvgSales_14d_vs_28d', 
            'Growth_AvgSales_28d_vs_56d', 'Growth_AvgSales_42d_vs_84d', 'SalesGrowth42dYearAgo', 
            'SalesPerCustomer7d', 'SalesPerCustomer14d', 'SalesPerCustomer28d', 'SalesPerCustomer42d', 
            'SalesPerCustomer56d', 'SalesPerCustomer84d', 'AvgSalesPerCustomer7d', 'AvgSalesPerCustomer14d', 
            'AvgSalesPerCustomer28d', 'AvgSalesPerCustomer42d', 'AvgSalesPerCustomer56d', 'AvgSalesPerCustomer84d', 
            'QtdCustomers7d', 'QtdCustomers14d', 'AvgCustomers7d', 'AvgCustomers14d', 'AvgCustomers28d', 
            'AvgCustomers42d', 'AvgCustomers56d', 'AvgCustomers84d', 'MinCustomers7d', 'MinCustomers14d', 
            'MinCustomers28d', 'MinCustomers42d', 'MinCustomers56d', 'MinCustomers84d', 'MaxCustomers7d', 
            'MaxCustomers14d', 'Growth_AvgCustomers_7d_vs_28d', 'Growth_AvgCustomers_14d_vs_28d', 
            'Growth_AvgCustomers_28d_vs_56d', 'Growth_AvgCustomers_42d_vs_84d'
        ]

    def fit(self, X, y=None):
        # CompetitionDistance median para imputação
        self.competition_distance_median_ = X['CompetitionDistance'].median()
        return self
    
    def transform(self, X):
        X = X.copy()  # para não alterar o original
        
        # SalesGrowth42dYearAgo_missing
        X["SalesGrowth42dYearAgo_missing"] = X["SalesGrowth42dYearAgo"].isna().astype(int)
        
        # CompetitionOpen_missing
        X["CompetitionOpen_missing"] = X["CompetitionOpen"].isna().astype(int)
        
        # CompetitionOpen: lógica dependente da CompetitionDistance
        def comp_open(row):
            if not pd.isna(row["CompetitionDistance"]):
                return 1
            elif pd.isna(row["CompetitionOpen"]):
                return 0
            else:
                return row["CompetitionOpen"]
        X["CompetitionOpen"] = X.apply(comp_open, axis=1)
        
        # CompetitionDistance_missing
        X["CompetitionDistance_missing"] = X["CompetitionDistance"].isna().astype(int)
        
        # CompetitionDistance imputação
        X["CompetitionDistance"] = X["CompetitionDistance"].fillna(self.competition_distance_median_)
        
        # lift_vars - indicar missing e valor -1 para nulos
        for col in self.lift_vars:
            X[f"{col}_missing"] = X[col].isna().astype(int)
            X[col] = X[col].fillna(-1)
        
        # cols_promo - nulo vira 0
        for col in self.cols_promo:
            if col in X.columns:
                X[col] = X[col].fillna(0)
        
        # cols_sales_customers - nulo vira 0
        for col in self.cols_sales_customers:
            if col in X.columns:
                X[col] = X[col].fillna(0)
        
        return X

cleaning_pipeline = NullImputer()
X_train = cleaning_pipeline.fit_transform(X_train)

In [21]:
nan_pct = X_train.isna().mean() * 100

nan_pct = nan_pct[nan_pct > 0].sort_values(ascending=False)

nan_pct_df = nan_pct.to_frame('Porcentagem_NaN').reset_index().rename(columns={'index': 'Coluna'})

# Exibir todas as colunas e linhas necessárias para visualização adequada
with pd.option_context('display.max_columns', None, 'display.width', 1000, 'display.max_rows', None):
    display(nan_pct_df)

,Coluna,Porcentagem_NaN
